In [1]:
import duckdb


In [2]:
con = duckdb.connect("../data/processed/revenue.duckdb", read_only=True)

In [3]:
print("=== Duplicados exactos ===")
print(con.execute("""
    SELECT *, COUNT(*) AS n
    FROM raw_transactions
    GROUP BY ALL
    HAVING COUNT(*) > 1
    ORDER BY n DESC
    LIMIT 10
""").df())

=== Duplicados exactos ===
  Invoice StockCode                          Description  Quantity  \
0  555524     22698       PINK REGENCY TEACUP AND SAUCER         1   
1  555524     22697      GREEN REGENCY TEACUP AND SAUCER         1   
2  537224     70007            HI TEC ALPINE HAND WARMER         1   
3  572861     22775  PURPLE DRAWERKNOB ACRYLIC EDWARDIAN        12   
4  537265     79000                   MOROCCAN TEA GLASS        12   
5  541266     21755             LOVE BUILDING BLOCK WORD         1   
6  496431     84826       ASSTD DESIGN 3D PAPER STICKERS         1   
7  537781     21098                CHRISTMAS TOILET ROLL         1   
8  537154     22727            ALARM CLOCK BAKELIKE RED          1   
9  537144     22086      PAPER CHAIN KIT 50'S CHRISTMAS          1   

          InvoiceDate  Price  Customer ID         Country   n  
0 2011-06-05 11:37:00   2.95      16923.0  United Kingdom  20  
1 2011-06-05 11:37:00   2.95      16923.0  United Kingdom  12  
2 2010-12-

In [4]:
print("\n=== ¿Los duplicados exactos comparten Invoice? ===")
print(con.execute("""
    WITH dups AS (
        SELECT Invoice, StockCode, Quantity, Price, InvoiceDate, "Customer ID", Country, Description,
               COUNT(*) AS n
        FROM raw_transactions
        GROUP BY ALL
        HAVING COUNT(*) > 1
    )
    SELECT
        COUNT(*) AS grupos,
        SUM(n) AS filas_totales,
        SUM(n - 1) AS filas_sobrantes
    FROM dups
""").df())


=== ¿Los duplicados exactos comparten Invoice? ===
   grupos  filas_totales  filas_sobrantes
0   32907        67242.0          34335.0


In [5]:
# 3. ¿Hay casos donde el mismo Invoice + StockCode aparece con distinta Quantity/Price?
#    (Esto indicaría líneas legítimas repetidas, no duplicados de exportación)
print("\n=== Mismo Invoice + StockCode con distinta Quantity o Price ===")
print(con.execute("""
    SELECT Invoice, StockCode, COUNT(*) AS n,
           COUNT(DISTINCT Quantity) AS q_distintas,
           COUNT(DISTINCT Price) AS p_distintas
    FROM raw_transactions
    GROUP BY Invoice, StockCode
    HAVING COUNT(*) > 1
    ORDER BY n DESC
    LIMIT 10
""").df())

con.close()


=== Mismo Invoice + StockCode con distinta Quantity o Price ===
   Invoice StockCode   n  q_distintas  p_distintas
0   555524     22698  20            1            1
1  C544580         S  16            1           15
2   555524     22697  12            1            1
3  C514024         M  10            3            8
4   536412     21448  10            2            1
5   537224     70007  10            1            1
6   537646     22585  10            3            1
7   502660     17021   9            3            1
8   537202     22147   8            2            1
9   536464     22866   8            2            1
